# 4. Numerical Solution of Linear Systems

Solve $Ax = b$ where $A$ is an $n \times n$ matrix. This notebook covers:
- **Gaussian elimination** with partial pivoting
- **LU decomposition**
- Iterative methods: **Jacobi** and **Gauss-Seidel**
- Convergence criteria and comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import lu

%matplotlib inline
np.set_printoptions(precision=6, suppress=True)

# Diagonally dominant test system (guarantees iterative convergence)
A = np.array([[10, -1,  2,  0],
              [-1, 11, -1,  3],
              [ 2, -1, 10, -1],
              [ 0,  3, -1,  8]], dtype=float)
b = np.array([6, 25, -11, 15], dtype=float)
x_exact = np.linalg.solve(A, b)
print(f"Exact solution: {x_exact}")

## 4.2 Gaussian Elimination with Partial Pivoting

Forward elimination reduces $[A|b]$ to upper triangular form, then back substitution finds $x$.

In [ ]:
def gauss_elimination(A, b):
    n = len(b)
    Ab = np.hstack([A.copy(), b.reshape(-1, 1)])
    for k in range(n):
        max_idx = np.argmax(np.abs(Ab[k:, k])) + k
        Ab[[k, max_idx]] = Ab[[max_idx, k]]
        for i in range(k+1, n):
            factor = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= factor * Ab[k, k:]
    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (Ab[i, -1] - np.dot(Ab[i, i+1:n], x[i+1:])) / Ab[i, i]
    return x

x_gauss = gauss_elimination(A, b)
print(f"Gauss solution: {x_gauss}")
print(f"Max error:      {np.max(np.abs(x_gauss - x_exact)):.2e}")

## 4.3 LU Decomposition

Factor $PA = LU$, then solve via forward + back substitution. Efficient for multiple right-hand sides.

In [ ]:
P, L, U = lu(A)
print("L (lower triangular):")
print(L)
print("\nU (upper triangular):")
print(U)
print(f"\nVerification ||PA - LU|| = {np.linalg.norm(P @ A - L @ U):.2e}")

y = np.linalg.solve(L, P @ b)
x_lu = np.linalg.solve(U, y)
print(f"\nLU solution: {x_lu}")

## 4.4 Jacobi and Gauss-Seidel Methods

**Jacobi**: $x^{(k+1)}_i = \frac{1}{a_{ii}}\left(b_i - \sum_{j \neq i} a_{ij} x^{(k)}_j\right)$ (uses old values)

**Gauss-Seidel**: same formula but uses the most recently computed values immediately. Generally converges faster.

In [ ]:
def jacobi(A, b, x0=None, tol=1e-10, max_iter=200):
    n = len(b)
    x = np.zeros(n) if x0 is None else x0.copy()
    D = np.diag(A)
    R = A - np.diag(D)
    errors = []
    for k in range(max_iter):
        x_new = (b - R @ x) / D
        err = np.linalg.norm(x_new - x, ord=np.inf)
        errors.append(err)
        if err < tol:
            return x_new, errors
        x = x_new
    return x, errors

def gauss_seidel(A, b, x0=None, tol=1e-10, max_iter=200):
    n = len(b)
    x = np.zeros(n) if x0 is None else x0.copy()
    errors = []
    for k in range(max_iter):
        x_old = x.copy()
        for i in range(n):
            s = b[i] - np.dot(A[i, :i], x[:i]) - np.dot(A[i, i+1:], x[i+1:])
            x[i] = s / A[i, i]
        err = np.linalg.norm(x - x_old, ord=np.inf)
        errors.append(err)
        if err < tol:
            return x, errors
    return x, errors

x_jac, errors_jac = jacobi(A, b)
x_gs, errors_gs = gauss_seidel(A, b)
print(f"Jacobi:       {x_jac} ({len(errors_jac)} iters)")
print(f"Gauss-Seidel: {x_gs} ({len(errors_gs)} iters)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(range(len(errors_jac)), errors_jac, 'o-', markersize=3, label='Jacobi')
ax.semilogy(range(len(errors_gs)), errors_gs, 's-', markersize=3, label='Gauss-Seidel')
ax.set_xlabel('Iteration')
ax.set_ylabel('$||x^{(k+1)} - x^{(k)}||_\\infty$')
ax.set_title('Convergence of Iterative Methods')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

| Method | Type | Complexity | When to use |
|--------|------|-----------|-------------|
| Gaussian Elimination | Direct | $O(n^3)$ | Small-medium dense systems |
| LU Decomposition | Direct | $O(n^3)$ | Multiple right-hand sides |
| Jacobi | Iterative | $O(n^2)$/step | Large sparse diag-dominant systems |
| Gauss-Seidel | Iterative | $O(n^2)$/step | Faster than Jacobi, same conditions |

- Direct methods give exact solutions (up to rounding) in a fixed number of operations
- Iterative methods are preferred for **large sparse** systems
- **Diagonal dominance** or positive definiteness guarantees iterative convergence